# 📈 Mini-Aladdin AI Trading System
**Your personal trading backtest and signal dashboard.**

## How to use:
1. Run each cell top to bottom (click ▶️ or press Shift+Enter)
2. Wait for each cell to finish before running the next
3. The dashboard link will appear at the bottom — click it!

---

In [ ]:
# Step 1: Install all required packages
print('Installing packages... this takes 2-3 minutes')
!pip install -q yfinance streamlit plotly pandas numpy ta requests python-dotenv pyngrok
print('✅ All packages installed!')

In [ ]:
# Step 2: Write strategy file
strategy_code = '''
import abc
import pandas as pd
import ta

BUY = 1
SELL = -1
HOLD = 0

class Strategy(abc.ABC):
    def __init__(self, **params):
        self.params = params
    @property
    @abc.abstractmethod
    def name(self): pass
    @abc.abstractmethod
    def generate_signals(self, df): pass
    def _validate_df(self, df):
        return df.copy()

class MovingAverageCrossover(Strategy):
    def __init__(self, fast_period=10, slow_period=50):
        super().__init__(fast_period=fast_period, slow_period=slow_period)
        self.fast_period = fast_period
        self.slow_period = slow_period
    @property
    def name(self): return f"MA Crossover ({self.fast_period}/{self.slow_period})"
    def generate_signals(self, df):
        df = self._validate_df(df)
        df["fast_ma"] = ta.trend.ema_indicator(df["Close"], window=self.fast_period)
        df["slow_ma"] = ta.trend.ema_indicator(df["Close"], window=self.slow_period)
        df["signal"] = HOLD
        bullish = (df["fast_ma"] > df["slow_ma"]) & (df["fast_ma"].shift(1) <= df["slow_ma"].shift(1))
        bearish = (df["fast_ma"] < df["slow_ma"]) & (df["fast_ma"].shift(1) >= df["slow_ma"].shift(1))
        df.loc[bullish, "signal"] = BUY
        df.loc[bearish, "signal"] = SELL
        df.dropna(subset=["fast_ma","slow_ma"], inplace=True)
        return df

class RSIMeanReversion(Strategy):
    def __init__(self, rsi_period=14, oversold=30, overbought=70):
        super().__init__(rsi_period=rsi_period, oversold=oversold, overbought=overbought)
        self.rsi_period = rsi_period
        self.oversold = oversold
        self.overbought = overbought
    @property
    def name(self): return f"RSI ({self.rsi_period}, {self.oversold}/{self.overbought})"
    def generate_signals(self, df):
        df = self._validate_df(df)
        df["rsi"] = ta.momentum.rsi(df["Close"], window=self.rsi_period)
        df["signal"] = HOLD
        df.loc[df["rsi"] < self.oversold, "signal"] = BUY
        df.loc[df["rsi"] > self.overbought, "signal"] = SELL
        df.dropna(subset=["rsi"], inplace=True)
        return df

class MACDStrategy(Strategy):
    def __init__(self, fast=12, slow=26, signal=9):
        super().__init__(fast=fast, slow=slow, signal=signal)
        self.fast = fast
        self.slow = slow
        self.signal_period = signal
    @property
    def name(self): return f"MACD ({self.fast}/{self.slow}/{self.signal_period})"
    def generate_signals(self, df):
        df = self._validate_df(df)
        macd = ta.trend.MACD(df["Close"], window_fast=self.fast, window_slow=self.slow, window_sign=self.signal_period)
        df["macd"] = macd.macd()
        df["macd_signal"] = macd.macd_signal()
        df["signal"] = HOLD
        bullish = (df["macd"] > df["macd_signal"]) & (df["macd"].shift(1) <= df["macd_signal"].shift(1))
        bearish = (df["macd"] < df["macd_signal"]) & (df["macd"].shift(1) >= df["macd_signal"].shift(1))
        df.loc[bullish, "signal"] = BUY
        df.loc[bearish, "signal"] = SELL
        df.dropna(subset=["macd","macd_signal"], inplace=True)
        return df

STRATEGY_REGISTRY = {
    "MovingAverageCrossover": {"class": MovingAverageCrossover, "default_params": {"fast_period": 10, "slow_period": 50}, "param_defs": {"fast_period": {"type": "int", "min": 2, "max": 50, "default": 10}, "slow_period": {"type": "int", "min": 10, "max": 200, "default": 50}}},
    "RSIMeanReversion": {"class": RSIMeanReversion, "default_params": {"rsi_period": 14, "oversold": 30, "overbought": 70}, "param_defs": {"rsi_period": {"type": "int", "min": 2, "max": 30, "default": 14}, "oversold": {"type": "int", "min": 10, "max": 40, "default": 30}, "overbought": {"type": "int", "min": 60, "max": 90, "default": 70}}},
    "MACDStrategy": {"class": MACDStrategy, "default_params": {"fast": 12, "slow": 26, "signal": 9}, "param_defs": {"fast": {"type": "int", "min": 3, "max": 20, "default": 12}, "slow": {"type": "int", "min": 10, "max": 50, "default": 26}, "signal": {"type": "int", "min": 3, "max": 20, "default": 9}}},
}

def build_strategy(name, **params):
    entry = STRATEGY_REGISTRY[name]
    merged = {**entry["default_params"], **params}
    return entry["class"](**merged)
'''

with open('strategy_simple.py', 'w') as f:
    f.write(strategy_code)
print('✅ Strategy file created!')

In [ ]:
# Step 3: Write backtest file
backtest_code = '''
import pandas as pd
import numpy as np
import yfinance as yf
from dataclasses import dataclass
from strategy_simple import Strategy, build_strategy, STRATEGY_REGISTRY, BUY, SELL

@dataclass
class BacktestResult:
    symbol: str
    strategy_name: str
    start_date: str
    end_date: str
    initial_capital: float
    equity_curve: pd.Series
    trades: pd.DataFrame
    metrics: dict
    signals_df: pd.DataFrame

def fetch_data(symbol, start, end):
    df = yf.Ticker(symbol).history(start=start, end=end, auto_adjust=True)
    if df.empty: raise ValueError(f"No data for {symbol}")
    return df[["Open","High","Low","Close","Volume"]].copy()

def run_backtest(symbol, strategy, start_date, end_date, initial_capital=10000.0, commission=0.001, slippage=0.001):
    df = fetch_data(symbol, start_date, end_date)
    signals_df = strategy.generate_signals(df)
    cash, shares, equity_curve, trades = initial_capital, 0.0, [], []
    entry_price, entry_date = 0.0, None
    for date, row in signals_df.iterrows():
        price = row["Close"]
        portfolio_value = cash + shares * price
        equity_curve.append({"date": date, "value": portfolio_value})
        if row["signal"] == BUY and shares == 0 and cash > 0:
            buy_price = price * (1 + slippage)
            shares = (cash * (1 - commission)) / buy_price
            entry_price, entry_date, cash = buy_price, date, 0.0
        elif row["signal"] == SELL and shares > 0:
            sell_price = price * (1 - slippage)
            proceeds = shares * sell_price * (1 - commission)
            trades.append({"Entry Date": entry_date, "Exit Date": date, "Entry Price": round(entry_price,4), "Exit Price": round(sell_price,4), "PnL": round(proceeds - shares*entry_price,2), "Return %": round((sell_price-entry_price)/entry_price*100,2)})
            cash, shares, entry_price, entry_date = proceeds, 0.0, 0.0, None
    equity_series = pd.Series([e["value"] for e in equity_curve], index=[e["date"] for e in equity_curve], name="Portfolio Value")
    trades_df = pd.DataFrame(trades) if trades else pd.DataFrame()
    return BacktestResult(symbol=symbol, strategy_name=strategy.name, start_date=start_date, end_date=end_date, initial_capital=initial_capital, equity_curve=equity_series, trades=trades_df, metrics=compute_metrics(equity_series, trades_df, initial_capital), signals_df=signals_df)

def compute_metrics(equity, trades, initial_capital):
    final_value = float(equity.iloc[-1])
    total_return = (final_value - initial_capital) / initial_capital * 100
    n_years = (equity.index[-1] - equity.index[0]).days / 365.25
    cagr = ((final_value/initial_capital)**(1/n_years)-1)*100 if n_years > 0 else 0.0
    dr = equity.pct_change().dropna()
    sharpe = float(dr.mean()/dr.std()*np.sqrt(252)) if dr.std() != 0 else 0.0
    max_dd = float(((equity - equity.cummax())/equity.cummax()*100).min())
    total_trades = len(trades)
    if total_trades > 0:
        winners = trades[trades["PnL"] > 0]
        losers = trades[trades["PnL"] <= 0]
        win_rate = len(winners)/total_trades*100
        gp = winners["PnL"].sum() if len(winners) > 0 else 0
        gl = abs(losers["PnL"].sum()) if len(losers) > 0 else 0
        pf = round(gp/gl,3) if gl > 0 else 0.0
    else:
        win_rate, pf = 0.0, 0.0
    return {"total_return": round(total_return,2), "cagr": round(cagr,2), "sharpe_ratio": round(sharpe,3), "max_drawdown": round(max_dd,2), "win_rate": round(win_rate,2), "profit_factor": pf, "total_trades": total_trades, "final_value": round(final_value,2), "initial_capital": round(initial_capital,2)}

def compare_strategies(symbol, start_date, end_date, initial_capital=10000.0):
    results = []
    for name in STRATEGY_REGISTRY:
        try:
            result = run_backtest(symbol, build_strategy(name), start_date, end_date, initial_capital)
            results.append(result)
        except: pass
    return sorted(results, key=lambda r: r.metrics.get("sharpe_ratio",-999), reverse=True)

def results_to_df(results):
    rows = [{"Strategy": r.strategy_name, **r.metrics} for r in results]
    return pd.DataFrame(rows).set_index("Strategy")
'''

with open('backtest_simple.py', 'w') as f:
    f.write(backtest_code)
print('✅ Backtest file created!')

In [ ]:
# Step 4: Write the dashboard app
app_code = '''
from datetime import date, timedelta
import plotly.graph_objects as go
import streamlit as st
import yfinance as yf
from strategy_simple import STRATEGY_REGISTRY, build_strategy, BUY, SELL
from backtest_simple import run_backtest, compare_strategies, results_to_df

st.set_page_config(page_title="Mini-Aladdin", page_icon="📈", layout="wide")
st.sidebar.title("📈 Mini-Aladdin")
st.sidebar.caption("Your Personal Trading System")
st.sidebar.divider()
symbol = st.sidebar.text_input("Stock Ticker", value="AAPL").upper().strip()
start_date = st.sidebar.date_input("Start Date", value=date.today() - timedelta(days=3*365))
end_date = st.sidebar.date_input("End Date", value=date.today() - timedelta(days=1))
strategy_name = st.sidebar.selectbox("Strategy", list(STRATEGY_REGISTRY.keys()))
st.sidebar.subheader("Parameters")
param_defs = STRATEGY_REGISTRY[strategy_name]["param_defs"]
default_params = STRATEGY_REGISTRY[strategy_name]["default_params"]
user_params = {}
for param, pdef in param_defs.items():
    user_params[param] = st.sidebar.slider(param, pdef["min"], pdef["max"], default_params[param])
st.sidebar.divider()
capital = st.sidebar.number_input("Capital ($)", min_value=100, value=10000, step=500)
run_btn = st.sidebar.button("🚀 Run Backtest", type="primary", use_container_width=True)
tab1, tab2, tab3 = st.tabs(["📊 Backtest", "⚖️ Compare All", "⚡ Live Quote"])
with tab1:
    st.header("Backtest Results")
    if run_btn:
        with st.spinner("Running backtest..."):
            try:
                strategy = build_strategy(strategy_name, **user_params)
                result = run_backtest(symbol, strategy, str(start_date), str(end_date), capital)
                st.session_state["result"] = result
            except Exception as e:
                st.error(f"Error: {e}")
    result = st.session_state.get("result")
    if result is None:
        st.info("👈 Set your parameters and click Run Backtest")
    else:
        m = result.metrics
        c1, c2, c3, c4 = st.columns(4)
        c1.metric("Total Return", f"{m[chr(39)+'total_return'+chr(39)]}%")
        c2.metric("CAGR", f"{m[chr(39)+'cagr'+chr(39)]}%")
        c3.metric("Sharpe Ratio", f"{m[chr(39)+'sharpe_ratio'+chr(39)]}")
        c4.metric("Max Drawdown", f"{m[chr(39)+'max_drawdown'+chr(39)]}%")
        c5, c6, c7, c8 = st.columns(4)
        c5.metric("Win Rate", f"{m[chr(39)+'win_rate'+chr(39)]}%")
        c6.metric("Profit Factor", f"{m[chr(39)+'profit_factor'+chr(39)]}x")
        c7.metric("Total Trades", m[chr(39)+"total_trades"+chr(39)])
        c8.metric("Final Value", f"${m[chr(39)+'final_value'+chr(39)]:,.2f}")
        fig = go.Figure()
        fig.add_trace(go.Scatter(x=result.equity_curve.index, y=result.equity_curve.values, fill="tozeroy", fillcolor="rgba(0,212,170,0.1)", line=dict(color="#00d4aa", width=2), name="Portfolio"))
        fig.add_hline(y=capital, line_dash="dash", line_color="gray")
        fig.update_layout(title="Equity Curve", template="plotly_dark", height=350)
        st.plotly_chart(fig, use_container_width=True)
        df = result.signals_df
        fig2 = go.Figure()
        fig2.add_trace(go.Scatter(x=df.index, y=df["Close"], line=dict(color="#aaa"), name="Price"))
        buys = df[df["signal"] == BUY]
        sells = df[df["signal"] == SELL]
        fig2.add_trace(go.Scatter(x=buys.index, y=buys["Close"], mode="markers", marker=dict(color="lime", size=10, symbol="triangle-up"), name="BUY"))
        fig2.add_trace(go.Scatter(x=sells.index, y=sells["Close"], mode="markers", marker=dict(color="red", size=10, symbol="triangle-down"), name="SELL"))
        fig2.update_layout(title="Price + Signals", template="plotly_dark", height=320)
        st.plotly_chart(fig2, use_container_width=True)
        if not result.trades.empty:
            st.subheader("Trade Log")
            st.dataframe(result.trades, use_container_width=True)
with tab2:
    st.header("Compare All Strategies")
    if st.button("▶️ Run Comparison", type="primary"):
        with st.spinner("Running all strategies..."):
            try:
                results = compare_strategies(symbol, str(start_date), str(end_date), capital)
                st.session_state["compare"] = results
            except Exception as e:
                st.error(f"Error: {e}")
    compare = st.session_state.get("compare")
    if compare:
        st.dataframe(results_to_df(compare), use_container_width=True)
        fig = go.Figure()
        colors = ["#00d4aa", "#ff9f43", "#a29bfe"]
        for i, r in enumerate(compare):
            norm = (r.equity_curve / r.equity_curve.iloc[0] - 1) * 100
            fig.add_trace(go.Scatter(x=norm.index, y=norm.values, line=dict(color=colors[i], width=2), name=r.strategy_name))
        fig.update_layout(title="Normalized Return %", template="plotly_dark", height=400)
        st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("Click Run Comparison to see all strategies")
with tab3:
    st.header("⚡ Live Quote")
    q = st.text_input("Ticker", value="AAPL").upper().strip()
    if st.button("Fetch Quote", type="primary"):
        try:
            ticker = yf.Ticker(q)
            info = ticker.fast_info
            price = info.last_price
            prev = info.previous_close
            c1, c2 = st.columns(2)
            c1.metric("Price", f"${price:,.2f}", f"{(price-prev)/prev*100:+.2f}%")
            c2.metric("Change", f"${price-prev:+.2f}")
            hist = ticker.history(period="1mo")
            fig = go.Figure(go.Candlestick(x=hist.index, open=hist["Open"], high=hist["High"], low=hist["Low"], close=hist["Close"]))
            fig.update_layout(title=f"{q} Last 30 Days", template="plotly_dark", height=400, xaxis_rangeslider_visible=False)
            st.plotly_chart(fig, use_container_width=True)
        except Exception as e:
            st.error(f"Error: {e}")
'''

with open('app.py', 'w') as f:
    f.write(app_code)
print('✅ Dashboard file created!')

In [ ]:
# Step 5: Launch the dashboard!
print('🚀 Starting Mini-Aladdin dashboard...')
print('A public link will appear below in about 10 seconds.')
print('Click it to open your trading dashboard!')
print('')

from pyngrok import ngrok
import subprocess
import time

# Start streamlit in background
process = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501', '--server.headless', 'true'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

# Create public tunnel
public_url = ngrok.connect(8501)
print(f'✅ Your dashboard is live!')
print(f'👉 Click here: {public_url}')
print('')
print('Keep this tab open to keep the dashboard running.')